In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=33
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

In [2]:
model_file_base = "vit.mlir"
for bs in [1]:
    model_file = model_file_base + ".bs{}".format(bs)
    target_file = model_file + ".heuristic"
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --canonicalize \
    --cse > {target_file}
#!grep "global_load" -rnc {target_file}
#!grep "global_store" -rnc {target_file}

user to this value: tosa.reshape
user to this value: tensor.insert_slice
user to this value: tosa.add
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.add
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.reciprocal
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.reshape
user to this value: tosa.add
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costly, worthy staging
user to this value: tosa.add
this user is compute-costly, worthy staging
user to this value: tosa.add
this user is compute-costly, worthy staging
user to this value: tosa.mul
this user is compute-costl

In [ ]:
model_file = "vit.mlir"
for bs in [1, 2, 4, 8, 16]:
    source_file = model_file + ".bs{}.heuristic".format(bs)
    target_file = source_file + ".cpu.vmfb"
    print("compiling {} to {}".format(source_file, target_file))
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-hal-target-backends=llvm-cpu \
    --iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0 \
    --iree-opt-const-eval=1 \
    --iree-opt-const-expr-hoisting=1 \
    --iree-opt-numeric-precision-reduction=1 \
    --iree-llvmcpu-target-cpu-features=host \
    --iree-llvmcpu-enable-ukernels=all \
    --iree-llvmcpu-slp-vectorization=1 \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-llvmcpu-target-triple=x86_64-pc-linux-elf

compiling vit.mlir.bs1.heuristic to vit.mlir.bs1.heuristic.cpu.vmfb
compiling vit.mlir.bs2.heuristic to vit.mlir.bs2.heuristic.cpu.vmfb
Please report issues to https://github.com/openxla/iree/issues and include the crash backtrace.
Stack dump:
0.	Program arguments: /root/miniconda3/envs/albert-research-py310/lib/python3.10/site-packages/iree/compiler/tools/../_mlir_libs/iree-compile vit.mlir.bs2.heuristic -o vit.mlir.bs2.heuristic.cpu.vmfb --iree-hal-target-backends=llvm-cpu --iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0 --iree-opt-const-eval=1 --iree-opt-const-expr-hoisting=1 --iree-opt-numeric-precision-reduction=1 --iree-llvmcpu-target-cpu-features=host --iree-llvmcpu-enable-ukernels=all --iree-llvmcpu-slp-vectorization=1 --iree-hal-benchmark-dispatch-repeat-count=33 --iree-llvmcpu-target-triple=x86_64-pc-linux-elf


In [ ]:
import gc
model_file = "vit.mlir"
for bs in [1,2,4,8,16,32]:
    source_file = model_file + ".bs{}.heuristic".format(bs)
    target_file = source_file + ".cpu.vmfb"
    print("compiling {} to {}".format(source_file, target_file))
    #!iree-compile {source_file} \
    #-o {target_file} \
    #--iree-hal-target-backends=cuda \
    #--iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    #--iree-hal-cuda-llvm-target-arch=sm_86

    ragdoll_binary = load_executable(target_file)

    image = torch.randn(bs, 3, 224, 224)
    image_np = image.detach().cpu().numpy()
    image_t = torch.randn(bs, 224, 224, 3)
    image_np_t = image_t.detach().cpu().numpy()
    model = models.vit_b_16().train(False)
    model.load_state_dict({k: torch.ones_like(v) * 0.23421 for k, v in model.state_dict().items()})
    output = model(image)
    grad = torch.randn_like(output)
    grad_np = grad.cpu().numpy()

    try:
        f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        print('ragdoll-opt1-gpu-forward in timeit: ', f1)
        b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT
        print('ragdoll-opt1-gpu-forward in timeit: ', b1)
        df = pd.concat([df, get_dataframe(f1, b1, "Ragdoll-Autodiff at batch-size = {}".format(bs))])
    except Exception as e:
        print(f"处理模型时出错，批量大小 {bs}: {e}")
    print(df)

In [ ]:
df.style.hide(axis="index")
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time Normalized (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")